# Q4 Attention Benchmark — Upgraded

Research question: **How does the model perform when critical information is buried among large amounts of irrelevant context?**

This upgraded version adds:
- larger configurable dossier profiles
- randomized target SECTION IDs
- distractor family metadata
- explicit retrieval + EPU + joint scoring
- buriedness failure-mode classification
- hard-failure / parse-failure detection
- partial save checkpoints


In [ ]:
import sys, subprocess
try:
    import scipy  # noqa: F401
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scipy", "-q"])

# ============================================================
# Q4 ATTENTION BENCHMARK — UPGRADED
# ------------------------------------------------------------
# Research question:
# How does the model perform when critical information is buried
# among large amounts of irrelevant context?
#
# Upgrades in this version:
# - larger configurable dossier profiles (compact / standard / extended)
# - randomized target SECTION ID instead of a fixed label
# - distractor families and difficulty metadata
# - optional multiple dossier variants per article
# - explicit retrieval + EPU + joint scoring
# - buriedness failure-mode classification
# - hard-failure / parse-failure detection
# - richer grounding and confidence diagnostics
# - partial saves after each condition checkpoint
# ============================================================

import json
import math
import re
import hashlib
from pathlib import Path
from typing import Dict, List, Any

import numpy as np
import pandas as pd
from scipy.stats import binomtest
from tqdm.auto import tqdm
from IPython.display import display

try:
    import kaggle_benchmarks as kbench
except Exception as e:
    raise RuntimeError("This notebook must run in Kaggle with kaggle_benchmarks available.") from e

# ---------------------------
# CONFIG
# ---------------------------
INPUT_CSV = "/kaggle/input/datasets/bigdataanalysis1/epu-800/EPU_benchmark_800_balanced.csv"

SEED = 42
N_ROWS = 600
MAX_CHARS = 4500
BOOT_N = 4000
SAVE_CHECKPOINT_EVERY = 50

MODEL_NAMES = [
    # "google/gemini-2.5-flash",
    # "google/gemini-2.5-pro",
    # "anthropic/claude-sonnet-4-6@default",
    # "deepseek-ai/deepseek-v3.2",
    "qwen/qwen3-235b-a22b-instruct-2507",
]

# Keep defaults near current cost. Increase only if you want stronger scientific coverage.
DOSSIER_PROFILE = "standard8"   # options: compact6, standard8, extended12
DOSSIER_VARIANTS_PER_ARTICLE = 1  # increase to 2–3 for stronger robustness checks
OUT_DIR = Path("/kaggle/working/epu_attention_q4_upgraded_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PROFILE_MAP = {
    "compact6": {"n_sections": 6, "positions": {"easy": 2, "medium": 3, "hard": 5, "very_hard": 6}},
    "standard8": {"n_sections": 8, "positions": {"easy": 2, "medium": 4, "hard": 6, "very_hard": 8}},
    "extended12": {"n_sections": 12, "positions": {"easy": 2, "medium": 5, "hard": 9, "very_hard": 12}},
}
CONDITIONS = ["easy", "medium", "hard", "very_hard"]
MISSING_SENTINEL = -1

DISTRACTOR_BANK = [
    {
        "family": "civic_notice",
        "difficulty": "easy",
        "text": (
            "Public library notice: weekend hours remain unchanged. A temporary exhibit on local architecture will open next month. "
            "Visitors are asked to use the east entrance during floor maintenance."
        ),
    },
    {
        "family": "weather_update",
        "difficulty": "easy",
        "text": (
            "Regional weather bulletin: light rain is expected in the north and cooler temperatures will continue through Thursday. "
            "No major service impacts are expected according to municipal staff."
        ),
    },
    {
        "family": "transport_notice",
        "difficulty": "easy",
        "text": (
            "Transit riders were advised of a platform change at the central station after scheduled track work. "
            "Officials said the timetable revision was routine and unrelated to any policy announcement."
        ),
    },
    {
        "family": "education_admin",
        "difficulty": "easy",
        "text": (
            "The university registrar released enrollment dates for summer courses and reminded students to confirm contact details. "
            "A separate note listed campus building access rules during exams."
        ),
    },
    {
        "family": "local_business",
        "difficulty": "medium",
        "text": (
            "A retail association said shoppers were more cautious this quarter, while store owners adjusted inventory and staffing. "
            "Merchants discussed consumer demand, but no government policy proposal was under consideration in the report."
        ),
    },
    {
        "family": "housing_market",
        "difficulty": "medium",
        "text": (
            "Real-estate agents described slower bidding activity and longer listing times in several neighbourhoods. "
            "The article mentioned mortgage sentiment and household budgets but did not discuss any uncertain policy action."
        ),
    },
    {
        "family": "company_earnings",
        "difficulty": "medium",
        "text": (
            "A manufacturing firm lowered its earnings outlook after shipping delays and softer overseas demand. "
            "Executives said they were monitoring costs closely but gave no indication that policy uncertainty was central to the decision."
        ),
    },
    {
        "family": "foreign_affairs",
        "difficulty": "hard",
        "text": (
            "Diplomats met for a new round of talks after tensions rose at the border. Traders noted market volatility and transport disruptions, "
            "but the dispatch contained no direct uncertainty about domestic economic policy decisions."
        ),
    },
    {
        "family": "election_campaign",
        "difficulty": "hard",
        "text": (
            "Campaign officials traded criticism over debate scheduling and fundraising totals. Analysts said the tone of the race could unsettle markets, "
            "yet the report did not identify any specific uncertain economic policy choice."
        ),
    },
    {
        "family": "commodity_markets",
        "difficulty": "hard",
        "text": (
            "Oil and grain prices moved sharply after supply news from abroad. Commentators discussed investor nerves and economic implications, "
            "but the article stopped short of describing uncertainty over a government economic policy decision."
        ),
    },
    {
        "family": "health_system",
        "difficulty": "medium",
        "text": (
            "Hospital administrators described staffing pressure and patient backlogs while outlining scheduling changes for clinics. "
            "The report focused on operations and service demand rather than uncertain economic policy."
        ),
    },
    {
        "family": "sports_business",
        "difficulty": "easy",
        "text": (
            "Club executives announced sponsorship renewals and venue upgrades ahead of next season. Supporters discussed ticket prices and travel demand, "
            "but the report was limited to team operations and local commerce."
        ),
    },
]

CONDITION_DIFFICULTY_HINT = {
    "easy": ["easy", "medium"],
    "medium": ["easy", "medium", "hard"],
    "hard": ["medium", "hard"],
    "very_hard": ["medium", "hard"],
}

# ---------------------------
# LOAD + CLEAN
# ---------------------------
df = pd.read_csv(INPUT_CSV)
required_cols = [
    "article_key",
    "content",
    "gold_epu",
    "gold_who",
    "gold_actions",
    "gold_effects",
    "mention_foreign",
    "mainly_foreign",
    "disagreement_flag",
    "newspaper",
    "year",
    "content_len",
]
for c in required_cols:
    if c not in df.columns:
        df[c] = np.nan

mask = (
    df["content"].notna()
    & (df["content"].astype(str).str.len() > 300)
    & df["gold_epu"].notna()
)
clean = df.loc[mask].copy()
clean["article_key"] = clean["article_key"].astype(str)
bad_key_mask = clean["article_key"].isin(["", "nan", "None"])
if bad_key_mask.any():
    clean.loc[bad_key_mask, "article_key"] = [f"generated_key_{i}" for i in range(int(bad_key_mask.sum()))]

clean["gold_epu"] = clean["gold_epu"].astype(int)
clean["gold_who"] = clean["gold_who"].fillna(MISSING_SENTINEL).astype(int)
clean["gold_actions"] = clean["gold_actions"].fillna(MISSING_SENTINEL).astype(int)
clean["gold_effects"] = clean["gold_effects"].fillna(MISSING_SENTINEL).astype(int)
clean["gold_mention_foreign"] = ((clean.get("gold_mention_foreign") if "gold_mention_foreign" in clean.columns else pd.Series(index=clean.index, dtype=float)).fillna(clean["mention_foreign"]).fillna(0) >= 0.5).astype(int)
clean["gold_mainly_foreign"] = ((clean.get("gold_mainly_foreign") if "gold_mainly_foreign" in clean.columns else pd.Series(index=clean.index, dtype=float)).fillna(clean["mainly_foreign"]).fillna(0) >= 0.5).astype(int)
clean["disagreement_flag"] = clean["disagreement_flag"].fillna(0).astype(int)
clean = clean.drop_duplicates(subset=["article_key"]).reset_index(drop=True)

print(f"Loaded rows: {len(df):,}")
print(f"Clean usable rows: {len(clean):,}")
print(clean["gold_epu"].value_counts().sort_index())
print(clean["disagreement_flag"].value_counts().sort_index())

# ---------------------------
# SAMPLE
# ---------------------------
def balanced_sample(data: pd.DataFrame, n: int, seed: int = 42) -> pd.DataFrame:
    pos = data[data["gold_epu"] == 1].copy()
    neg = data[data["gold_epu"] == 0].copy()
    if len(pos) == 0 or len(neg) == 0:
        return data.sample(min(n, len(data)), random_state=seed).reset_index(drop=True)
    n_pos = min(len(pos), n // 2)
    n_neg = min(len(neg), n - n_pos)
    pos_s = pos.sample(n=n_pos, random_state=seed)
    neg_s = neg.sample(n=n_neg, random_state=seed)
    out = pd.concat([pos_s, neg_s], ignore_index=True)
    out = out.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

pilot = balanced_sample(clean, N_ROWS, seed=SEED)
print("Pilot rows:", len(pilot))
display(pilot[["article_key", "gold_epu", "disagreement_flag", "content_len"]].head())

# ---------------------------
# HELPERS
# ---------------------------
def stable_hash(text: str) -> int:
    return int(hashlib.sha256(text.encode("utf-8")).hexdigest(), 16)


def smart_truncate(text: str, max_chars: int = MAX_CHARS):
    text = str(text)
    orig_len = len(text)
    if orig_len <= max_chars:
        return text, 0
    head = int(max_chars * 0.6)
    tail = max_chars - head - 20
    return text[:head] + "\n...[TRUNCATED MIDDLE]...\n" + text[-tail:], 1


def clean_snippet(text: str) -> str:
    return re.sub(r"\s+", " ", str(text)).strip()


def choose_distractor_indices(article_key: str, cond: str, variant_idx: int, n_needed: int) -> List[int]:
    allowed = CONDITION_DIFFICULTY_HINT[cond]
    candidate_idx = [i for i, x in enumerate(DISTRACTOR_BANK) if x["difficulty"] in allowed]
    # deterministic shuffle by article/condition/variant
    ordered = sorted(
        candidate_idx,
        key=lambda i: stable_hash(f"{article_key}|{cond}|{variant_idx}|{DISTRACTOR_BANK[i]['family']}"),
    )
    return ordered[:n_needed]


def section_ids_for_profile(profile_name: str) -> List[str]:
    n = PROFILE_MAP[profile_name]["n_sections"]
    return [f"SEC-{100 + 7*i}" for i in range(n)]


def build_dossier(article_key: str, content_for_eval: str, cond: str, variant_idx: int, profile_name: str = DOSSIER_PROFILE) -> Dict[str, Any]:
    profile = PROFILE_MAP[profile_name]
    n_sections = profile["n_sections"]
    target_position = profile["positions"][cond]  # 1-based
    n_distractors = n_sections - 1

    sec_ids = section_ids_for_profile(profile_name)
    # Randomize which actual SECTION ID is the target, while keeping the target position fixed.
    target_sid_idx = stable_hash(f"{article_key}|target_sid|{variant_idx}|{profile_name}") % len(sec_ids)
    target_section_id = sec_ids[target_sid_idx]
    other_ids = [x for x in sec_ids if x != target_section_id]

    dist_idx = choose_distractor_indices(article_key, cond, variant_idx, n_distractors)
    distractors = [DISTRACTOR_BANK[i] for i in dist_idx]

    sections = []
    other_id_ptr = 0
    dist_ptr = 0
    for pos in range(1, n_sections + 1):
        if pos == target_position:
            sections.append({
                "section_id": target_section_id,
                "section_type": "target",
                "family": "target_article",
                "difficulty": cond,
                "text": content_for_eval,
                "position": pos,
            })
        else:
            d = distractors[dist_ptr]
            dist_ptr += 1
            sections.append({
                "section_id": other_ids[other_id_ptr],
                "section_type": "distractor",
                "family": d["family"],
                "difficulty": d["difficulty"],
                "text": d["text"],
                "position": pos,
            })
            other_id_ptr += 1

    dossier_parts = []
    for s in sections:
        dossier_parts.append(
            f"SECTION ID {s['section_id']}\n"
            f"SECTION TYPE: {s['section_type']}\n"
            f"SECTION POSITION: {s['position']} of {n_sections}\n"
            f"TEXT START\n{s['text']}\nTEXT END"
        )
    dossier_text = "\n\n".join(dossier_parts)
    families = [s["family"] for s in sections if s["section_type"] == "distractor"]

    return {
        "target_section_id": target_section_id,
        "target_position": target_position,
        "n_sections": n_sections,
        "dossier_text": dossier_text,
        "distractor_families": "|".join(families),
        "target_position_norm": target_position / n_sections,
    }


def build_buried_prompt(target_section_id: str, dossier_text: str) -> str:
    return (
        "You are given a dossier with multiple sectioned articles.\n\n"
        f"Evaluate ONLY the article in SECTION ID {target_section_id}.\n"
        "Ignore every other section, even if it contains politics, markets, war, elections, prices, or uncertainty language.\n\n"
        "Task:\n"
        f"1. Identify SECTION ID {target_section_id}.\n"
        f"2. Use only that target section to decide whether SECTION ID {target_section_id} expresses ECONOMIC POLICY UNCERTAINTY (EPU).\n\n"
        "Labeling rule:\n"
        "- EPU = 1 only if the target section discusses uncertainty about government economic policy, policy decisions, possible policy actions, political outcomes affecting policy, or the economic effects of policy.\n"
        "- EPU = 0 for general economics, elections, markets, war, foreign events, or uncertainty language unless the uncertainty is specifically tied to economic policy.\n\n"
        "IMPORTANT:\n"
        "- The relevant section may appear anywhere in the dossier.\n"
        "- Use ONLY the named target SECTION ID.\n"
        "- Do not use any other section as evidence.\n"
        "- focus_excerpt must come from the target section only.\n"
        "- Return ONLY valid JSON. No markdown. No explanation. No code fence.\n\n"
        "Return exactly these keys:\n"
        "{\n"
        f'  "target_section_id": "{target_section_id}",\n'
        '  "epu": 0 or 1,\n'
        '  "confidence": number between 0 and 1,\n'
        '  "focus_excerpt": "short quote from the target section only"\n'
        "}\n\n"
        "DOSSIER START\n"
        f"{dossier_text}\n"
        "DOSSIER END"
    ).strip()


def default_out() -> Dict[str, Any]:
    return {
        "target_section_id": "",
        "epu": None,
        "confidence": np.nan,
        "focus_excerpt": "",
        "json_valid": 0,
        "parse_failure": 1,
        "raw_text": "",
    }


def parse_llm_json(raw: Any) -> Dict[str, Any]:
    out = default_out()
    text = str(raw)
    out["raw_text"] = text[:4000]
    if not text or text.strip() == "":
        return out

    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?", "", cleaned).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()

    # first try full parse
    obj = None
    try:
        obj = json.loads(cleaned)
    except Exception:
        m = re.search(r"\{.*\}", cleaned, flags=re.S)
        if m:
            try:
                obj = json.loads(m.group(0))
            except Exception:
                obj = None

    if not isinstance(obj, dict):
        return out

    out["json_valid"] = 1
    out["parse_failure"] = 0
    out["target_section_id"] = str(obj.get("target_section_id", "")).strip()
    out["focus_excerpt"] = clean_snippet(str(obj.get("focus_excerpt", "")))[:300]
    try:
        if obj.get("epu") is not None:
            out["epu"] = int(obj.get("epu"))
    except Exception:
        out["epu"] = None
    try:
        c = float(obj.get("confidence", np.nan))
        out["confidence"] = min(1.0, max(0.0, c))
    except Exception:
        out["confidence"] = np.nan
    return out


def model_json_call(llm, prompt: str) -> Dict[str, Any]:
    try:
        raw = llm.prompt(prompt)
        return parse_llm_json(raw)
    except Exception as e:
        out = default_out()
        out["raw_text"] = f"ERROR: {str(e)[:3500]}"
        return out


def maybe_int(x):
    try:
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return np.nan
        return int(x)
    except Exception:
        return np.nan


def safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan


def binary_match(pred, gold):
    try:
        if pred is None or (isinstance(pred, float) and np.isnan(pred)):
            return np.nan
        return float(int(pred) == int(gold))
    except Exception:
        return np.nan


def section_match(pred_sid: str, gold_sid: str):
    p = str(pred_sid or "").strip()
    g = str(gold_sid or "").strip()
    if p == "" or g == "":
        return np.nan
    return float(p == g)


def excerpt_in_target(excerpt: str, target_text: str) -> int:
    ex = clean_snippet(excerpt)
    tgt = clean_snippet(target_text)
    if ex == "":
        return 0
    if ex in tgt:
        return 1
    if len(ex) < 25:
        return 0
    tokens = set(re.findall(r"\w+", ex.lower()))
    tgt_tokens = set(re.findall(r"\w+", tgt.lower()))
    overlap = len(tokens & tgt_tokens) / max(1, len(tokens))
    return int(overlap >= 0.8)


def wilson_ci(k: int, n: int, z: float = 1.96):
    if n == 0:
        return (np.nan, np.nan)
    phat = k / n
    denom = 1 + z**2 / n
    center = (phat + z**2 / (2*n)) / denom
    half = z * math.sqrt((phat*(1-phat) + z**2/(4*n)) / n) / denom
    return center - half, center + half


def bootstrap_mean_ci(x: np.ndarray, n_boot: int = BOOT_N, seed: int = 42):
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    means = []
    for _ in range(n_boot):
        sample = rng.choice(x, size=len(x), replace=True)
        means.append(np.mean(sample))
    lo, hi = np.quantile(means, [0.025, 0.975])
    return float(lo), float(hi)


def bootstrap_diff_ci(a: np.ndarray, b: np.ndarray, n_boot: int = BOOT_N, seed: int = 42):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    mask = ~(np.isnan(a) | np.isnan(b))
    a = a[mask]
    b = b[mask]
    if len(a) == 0:
        return (np.nan, np.nan)
    d = b - a
    rng = np.random.default_rng(seed)
    means = []
    for _ in range(n_boot):
        sample = rng.choice(d, size=len(d), replace=True)
        means.append(np.mean(sample))
    lo, hi = np.quantile(means, [0.025, 0.975])
    return float(lo), float(hi)


def exact_mcnemar(a: np.ndarray, b: np.ndarray):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    mask = ~(np.isnan(a) | np.isnan(b))
    a = a[mask]
    b = b[mask]
    n01 = int(np.sum((a == 0) & (b == 1)))
    n10 = int(np.sum((a == 1) & (b == 0)))
    n = n01 + n10
    if n == 0:
        return {"mcnemar_p": 1.0, "discordant_n": 0, "n01": 0, "n10": 0}
    p = binomtest(min(n01, n10), n=n, p=0.5, alternative="two-sided").pvalue
    return {"mcnemar_p": float(p), "discordant_n": int(n), "n01": int(n01), "n10": int(n10)}


def classify_buried_failure(overall_drop_pp, retrieval_drop_pp, pos_drop_pp, neg_drop_pp, parse_spike_pp):
    vals = [overall_drop_pp, retrieval_drop_pp, pos_drop_pp, neg_drop_pp]
    if any(pd.isna(v) for v in vals):
        return "mixed"
    if parse_spike_pp >= 10:
        return "hard_failure_or_parse_break"
    if overall_drop_pp <= 1.0 and retrieval_drop_pp <= 2.0 and abs(pos_drop_pp) <= 2.0 and abs(neg_drop_pp) <= 2.0:
        return "buried_robust"
    if retrieval_drop_pp >= 5.0 and overall_drop_pp >= 4.0:
        return "retrieval_plus_grounding_failure"
    if retrieval_drop_pp >= 5.0 and overall_drop_pp < 4.0:
        return "primary_retrieval_loss"
    if overall_drop_pp >= 4.0 and retrieval_drop_pp < 5.0:
        if neg_drop_pp - pos_drop_pp >= 4.0:
            return "pro_epu_buried_drift"
        if pos_drop_pp - neg_drop_pp >= 4.0:
            return "conservative_buried_shift"
        return "grounding_or_judgment_loss"
    if neg_drop_pp - pos_drop_pp >= 4.0:
        return "pro_epu_buried_drift"
    if pos_drop_pp - neg_drop_pp >= 4.0:
        return "conservative_buried_shift"
    return "mixed"

# ---------------------------
# DOSSIER PREP
# ---------------------------
prepared_rows = []
for row in pilot.to_dict(orient="records"):
    content_for_eval, was_truncated = smart_truncate(row["content"], MAX_CHARS)
    rec = {
        **row,
        "content_for_eval": content_for_eval,
        "was_truncated": int(was_truncated),
        "orig_content_len": len(str(row["content"])),
        "content_for_eval_len": len(content_for_eval),
    }
    for variant_idx in range(1, DOSSIER_VARIANTS_PER_ARTICLE + 1):
        rec[f"variant_{variant_idx}_id"] = variant_idx
        for cond in CONDITIONS:
            d = build_dossier(row["article_key"], content_for_eval, cond, variant_idx, profile_name=DOSSIER_PROFILE)
            for k, v in d.items():
                rec[f"v{variant_idx}_{cond}_{k}"] = v
    prepared_rows.append(rec)

prepared = pd.DataFrame(prepared_rows)
prepared_path = OUT_DIR / "pilot_rows_q4_upgraded.csv"
prepared.to_csv(prepared_path, index=False)
print(f"Saved pilot rows -> {prepared_path}")
display(prepared.head(2))

# ---------------------------
# RUN MODELS
# ---------------------------
model_handles = {name: kbench.llms[name] for name in MODEL_NAMES}
records = []
status_rows = []

for model_name in MODEL_NAMES:
    llm = model_handles[model_name]
    print(f"\nRunning {model_name} ...")
    done_rows = 0
    try:
        for row in tqdm(prepared.to_dict(orient="records"), total=len(prepared), desc=model_name):
            for variant_idx in range(1, DOSSIER_VARIANTS_PER_ARTICLE + 1):
                rec = {
                    "llm_name": model_name,
                    "article_key": row["article_key"],
                    "gold_epu": row["gold_epu"],
                    "disagreement_flag": row["disagreement_flag"],
                    "newspaper": row.get("newspaper", ""),
                    "year": row.get("year", np.nan),
                    "content_len": row.get("content_len", np.nan),
                    "orig_content_len": row.get("orig_content_len", np.nan),
                    "content_for_eval_len": row.get("content_for_eval_len", np.nan),
                    "was_truncated": row.get("was_truncated", 0),
                    "content_for_eval": row["content_for_eval"],
                    "variant_id": variant_idx,
                    "dossier_profile": DOSSIER_PROFILE,
                }
                for cond in CONDITIONS:
                    prefix = f"v{variant_idx}_{cond}_"
                    target_section_id = row[f"{prefix}target_section_id"]
                    dossier_text = row[f"{prefix}dossier_text"]
                    out = model_json_call(llm, build_buried_prompt(target_section_id, dossier_text))

                    rec[f"{cond}_target_section_id"] = target_section_id
                    rec[f"{cond}_target_position"] = row[f"{prefix}target_position"]
                    rec[f"{cond}_target_position_norm"] = row[f"{prefix}target_position_norm"]
                    rec[f"{cond}_n_sections"] = row[f"{prefix}n_sections"]
                    rec[f"{cond}_distractor_families"] = row[f"{prefix}distractor_families"]
                    rec[f"{cond}_target_section_pred"] = out["target_section_id"]
                    rec[f"{cond}_section_correct"] = section_match(out["target_section_id"], target_section_id)
                    rec[f"{cond}_epu_pred"] = maybe_int(out["epu"])
                    rec[f"{cond}_epu_correct"] = binary_match(out["epu"], row["gold_epu"])
                    rec[f"{cond}_joint_correct"] = (
                        float(rec[f"{cond}_section_correct"] == 1.0 and rec[f"{cond}_epu_correct"] == 1.0)
                        if not pd.isna(rec[f"{cond}_section_correct"]) and not pd.isna(rec[f"{cond}_epu_correct"])
                        else np.nan
                    )
                    rec[f"{cond}_confidence"] = safe_float(out["confidence"])
                    rec[f"{cond}_focus_excerpt"] = out["focus_excerpt"]
                    rec[f"{cond}_excerpt_in_target"] = excerpt_in_target(out["focus_excerpt"], row["content_for_eval"])
                    rec[f"{cond}_json_valid"] = int(out["json_valid"])
                    rec[f"{cond}_parse_failure"] = int(out.get("parse_failure", 1))
                    rec[f"{cond}_raw_json"] = out["raw_text"]
                rec["easy_to_vhard_hurt"] = int(
                    rec["easy_joint_correct"] == 1.0 and rec["very_hard_joint_correct"] == 0.0
                ) if not pd.isna(rec["easy_joint_correct"]) and not pd.isna(rec["very_hard_joint_correct"]) else 0
                records.append(rec)

            done_rows += 1
            if done_rows % SAVE_CHECKPOINT_EVERY == 0:
                pd.DataFrame(records).to_csv(OUT_DIR / "q4_results_wide_checkpoint.csv", index=False)
                pd.DataFrame(status_rows + [{"llm_name": model_name, "status": "running", "rows_completed": done_rows}]).to_csv(OUT_DIR / "run_status_q4_upgraded.csv", index=False)

        status_rows.append({"llm_name": model_name, "status": "ok", "rows_completed": done_rows})
    except Exception as e:
        status_rows.append({"llm_name": model_name, "status": f"error: {str(e)[:500]}", "rows_completed": done_rows})
        pd.DataFrame(records).to_csv(OUT_DIR / "q4_results_wide_checkpoint.csv", index=False)
        pd.DataFrame(status_rows).to_csv(OUT_DIR / "run_status_q4_upgraded.csv", index=False)
        raise

wide_df = pd.DataFrame(records)
wide_path = OUT_DIR / "q4_results_wide_upgraded.csv"
wide_df.to_csv(wide_path, index=False)
pd.DataFrame(status_rows).to_csv(OUT_DIR / "run_status_q4_upgraded.csv", index=False)
print(f"Saved wide results -> {wide_path}")

# ---------------------------
# LONG VERSION
# ---------------------------
long_records = []
for row in wide_df.to_dict(orient="records"):
    base = {
        "llm_name": row["llm_name"],
        "article_key": row["article_key"],
        "gold_epu": row["gold_epu"],
        "disagreement_flag": row["disagreement_flag"],
        "newspaper": row.get("newspaper", ""),
        "year": row.get("year", np.nan),
        "variant_id": row["variant_id"],
        "was_truncated": row["was_truncated"],
        "orig_content_len": row["orig_content_len"],
        "content_for_eval_len": row["content_for_eval_len"],
        "dossier_profile": row["dossier_profile"],
    }
    for cond in CONDITIONS:
        long_records.append({
            **base,
            "condition": cond,
            "target_section_id": row[f"{cond}_target_section_id"],
            "target_section_pred": row[f"{cond}_target_section_pred"],
            "section_correct": row[f"{cond}_section_correct"],
            "epu_pred": row[f"{cond}_epu_pred"],
            "epu_correct": row[f"{cond}_epu_correct"],
            "joint_correct": row[f"{cond}_joint_correct"],
            "confidence": row[f"{cond}_confidence"],
            "focus_excerpt": row[f"{cond}_focus_excerpt"],
            "excerpt_in_target": row[f"{cond}_excerpt_in_target"],
            "json_valid": row[f"{cond}_json_valid"],
            "parse_failure": row[f"{cond}_parse_failure"],
            "target_position": row[f"{cond}_target_position"],
            "target_position_norm": row[f"{cond}_target_position_norm"],
            "n_sections": row[f"{cond}_n_sections"],
            "distractor_families": row[f"{cond}_distractor_families"],
            "raw_json": row[f"{cond}_raw_json"],
        })
long_df = pd.DataFrame(long_records)
long_path = OUT_DIR / "q4_results_long_upgraded.csv"
long_df.to_csv(long_path, index=False)
print(f"Saved long results -> {long_path}")

# ---------------------------
# SUMMARIES
# ---------------------------
def summarize_condition(frame: pd.DataFrame, condition: str) -> Dict[str, Any]:
    g = frame[frame["condition"] == condition]
    acc = float(g["joint_correct"].mean()) if len(g) else np.nan
    acc_lo, acc_hi = bootstrap_mean_ci(g["joint_correct"].to_numpy())
    epu_acc = float(g["epu_correct"].mean()) if len(g) else np.nan
    sec_acc = float(g["section_correct"].mean()) if len(g) else np.nan
    parse_fail = float(g["parse_failure"].mean()) if len(g) else np.nan
    return {
        f"{condition}_acc": acc,
        f"{condition}_acc_ci_low": acc_lo,
        f"{condition}_acc_ci_high": acc_hi,
        f"{condition}_epu_only_acc": epu_acc,
        f"{condition}_section_retrieval_rate": sec_acc,
        f"{condition}_json_valid_rate": float(g["json_valid"].mean()) if len(g) else np.nan,
        f"{condition}_parse_failure_rate": parse_fail,
        f"{condition}_excerpt_in_target_rate": float(g["excerpt_in_target"].mean()) if len(g) else np.nan,
        f"{condition}_confidence_mean": float(g["confidence"].mean()) if len(g) else np.nan,
        f"{condition}_target_position_mean": float(g["target_position_norm"].mean()) if len(g) else np.nan,
    }


overall_rows = []
for model_name, g_all in long_df.groupby("llm_name"):
    rec = {"llm_name": model_name, "n_rows": int(g_all[["article_key", "variant_id"]].drop_duplicates().shape[0])}
    for cond in CONDITIONS:
        rec.update(summarize_condition(g_all, cond))

    easy = g_all[g_all["condition"] == "easy"].sort_values(["article_key", "variant_id"])
    vhard = g_all[g_all["condition"] == "very_hard"].sort_values(["article_key", "variant_id"])
    easy_acc = easy["joint_correct"].to_numpy()
    vhard_acc = vhard["joint_correct"].to_numpy()
    easy_ret = easy["section_correct"].to_numpy()
    vhard_ret = vhard["section_correct"].to_numpy()
    easy_pos = easy[easy["gold_epu"] == 1]["joint_correct"].to_numpy()
    vhard_pos = vhard[vhard["gold_epu"] == 1]["joint_correct"].to_numpy()
    easy_neg = easy[easy["gold_epu"] == 0]["joint_correct"].to_numpy()
    vhard_neg = vhard[vhard["gold_epu"] == 0]["joint_correct"].to_numpy()

    easy_to_vhard_drop_pp = 100.0 * (np.nanmean(vhard_acc) - np.nanmean(easy_acc)) if len(easy_acc) else np.nan
    ret_drop_pp = 100.0 * (np.nanmean(vhard_ret) - np.nanmean(easy_ret)) if len(easy_ret) else np.nan
    pos_drop_pp = 100.0 * (np.nanmean(vhard_pos) - np.nanmean(easy_pos)) if len(easy_pos) else np.nan
    neg_drop_pp = 100.0 * (np.nanmean(vhard_neg) - np.nanmean(easy_neg)) if len(easy_neg) else np.nan
    parse_spike_pp = 100.0 * (rec["very_hard_parse_failure_rate"] - rec["easy_parse_failure_rate"])
    rec["easy_to_very_hard_drop_pp"] = easy_to_vhard_drop_pp
    rec["easy_to_very_hard_retrieval_drop_pp"] = ret_drop_pp
    rec["easy_to_very_hard_pos_drop_pp"] = pos_drop_pp
    rec["easy_to_very_hard_neg_drop_pp"] = neg_drop_pp
    lo, hi = bootstrap_diff_ci(easy_acc, vhard_acc)
    rec["easy_to_very_hard_drop_ci_low"] = 100.0 * lo if not pd.isna(lo) else np.nan
    rec["easy_to_very_hard_drop_ci_high"] = 100.0 * hi if not pd.isna(hi) else np.nan
    rec["failure_mode"] = classify_buried_failure(-easy_to_vhard_drop_pp, -ret_drop_pp, -pos_drop_pp, -neg_drop_pp, parse_spike_pp)
    overall_rows.append(rec)

summary_overall = pd.DataFrame(overall_rows)
summary_overall_path = OUT_DIR / "q4_summary_overall_upgraded.csv"
summary_overall.to_csv(summary_overall_path, index=False)
display(summary_overall)
print(f"Saved overall summary -> {summary_overall_path}")

# ---------------------------
# RETRIEVAL TABLE
# ---------------------------
retrieval_rows = []
for (model_name, cond), g in long_df.groupby(["llm_name", "condition"]):
    retrieval_rows.append({
        "llm_name": model_name,
        "condition": cond,
        "n_rows": len(g),
        "section_retrieval_rate": float(g["section_correct"].mean()),
        "epu_only_acc": float(g["epu_correct"].mean()),
        "joint_acc": float(g["joint_correct"].mean()),
        "json_valid_rate": float(g["json_valid"].mean()),
        "parse_failure_rate": float(g["parse_failure"].mean()),
        "excerpt_in_target_rate": float(g["excerpt_in_target"].mean()),
        "confidence_mean": float(g["confidence"].mean()),
        "mean_target_position_norm": float(g["target_position_norm"].mean()),
    })
retrieval_table = pd.DataFrame(retrieval_rows)
retrieval_table_path = OUT_DIR / "q4_retrieval_table_upgraded.csv"
retrieval_table.to_csv(retrieval_table_path, index=False)
print(f"Saved retrieval table -> {retrieval_table_path}")
display(retrieval_table)

# ---------------------------
# PAIRWISE PROOF
# ---------------------------
def pairwise_summary(frame: pd.DataFrame, cond_a: str, cond_b: str, compare_name: str):
    a_df = frame[frame["condition"] == cond_a].sort_values(["article_key", "variant_id"])
    b_df = frame[frame["condition"] == cond_b].sort_values(["article_key", "variant_id"])
    a = a_df["joint_correct"].to_numpy(dtype=float)
    b = b_df["joint_correct"].to_numpy(dtype=float)
    pred_a = a_df["epu_pred"].to_numpy(dtype=float)
    pred_b = b_df["epu_pred"].to_numpy(dtype=float)

    mask = ~(np.isnan(a) | np.isnan(b))
    a = a[mask]
    b = b[mask]
    acc_a = float(np.mean(a)) if len(a) else np.nan
    acc_b = float(np.mean(b)) if len(b) else np.nan
    drop_pp = 100.0 * (acc_b - acc_a) if len(a) else np.nan
    d_lo, d_hi = bootstrap_diff_ci(a, b)

    flip_mask = ~(np.isnan(pred_a) | np.isnan(pred_b))
    flip_n = int(np.sum(flip_mask))
    flip_k = int(np.sum(pred_a[flip_mask] != pred_b[flip_mask])) if flip_n else 0
    flip_rate = flip_k / flip_n if flip_n else np.nan
    flip_lo, flip_hi = wilson_ci(flip_k, flip_n)

    hurt_k = int(np.sum((a == 1) & (b == 0))) if len(a) else 0
    help_k = int(np.sum((a == 0) & (b == 1))) if len(a) else 0
    mc = exact_mcnemar(a, b)

    return {
        "comparison": compare_name,
        "n_rows": int(len(a)),
        "acc_a": acc_a,
        "acc_b": acc_b,
        "drop_pp": drop_pp,
        "drop_ci_low": 100.0 * d_lo if not pd.isna(d_lo) else np.nan,
        "drop_ci_high": 100.0 * d_hi if not pd.isna(d_hi) else np.nan,
        "flip_rate": flip_rate,
        "flip_rate_ci_low": flip_lo,
        "flip_rate_ci_high": flip_hi,
        "hurt_rate": hurt_k / len(a) if len(a) else np.nan,
        "help_rate": help_k / len(a) if len(a) else np.nan,
        **mc,
    }

pairwise_rows = []
for model_name, g in long_df.groupby("llm_name"):
    for a, b in [("easy", "medium"), ("easy", "hard"), ("easy", "very_hard"), ("medium", "hard"), ("medium", "very_hard"), ("hard", "very_hard")]:
        pairwise_rows.append({"llm_name": model_name, **pairwise_summary(g, a, b, f"{a} -> {b}")})
pairwise_df = pd.DataFrame(pairwise_rows)
pairwise_path = OUT_DIR / "q4_pairwise_proof_upgraded.csv"
pairwise_df.to_csv(pairwise_path, index=False)
print(f"Saved pairwise proof -> {pairwise_path}")
display(pairwise_df)

# ---------------------------
# SUBGROUP SUMMARIES
# ---------------------------
def subgroup_summary(frame: pd.DataFrame, group_col: str, out_name: str):
    rows = []
    for (model_name, group_val), g in frame.groupby(["llm_name", group_col]):
        rec = {"llm_name": model_name, group_col: group_val, "n_rows": int(len(g[g["condition"] == "easy"]))}
        for cond in CONDITIONS:
            gg = g[g["condition"] == cond]
            rec[f"{cond}_acc"] = float(gg["joint_correct"].mean()) if len(gg) else np.nan
            rec[f"{cond}_epu_only_acc"] = float(gg["epu_correct"].mean()) if len(gg) else np.nan
            rec[f"{cond}_section_retrieval_rate"] = float(gg["section_correct"].mean()) if len(gg) else np.nan
        rows.append(rec)
    out = pd.DataFrame(rows)
    path = OUT_DIR / out_name
    out.to_csv(path, index=False)
    print(f"Saved {out_name} -> {path}")
    return out

summary_by_gold = subgroup_summary(long_df, "gold_epu", "q4_summary_by_gold_epu_upgraded.csv")
summary_by_dis = subgroup_summary(long_df, "disagreement_flag", "q4_summary_by_disagreement_upgraded.csv")
summary_by_trunc = subgroup_summary(long_df, "was_truncated", "q4_summary_by_truncation_upgraded.csv")

# ---------------------------
# HARDEST HARMED ROWS
# ---------------------------
wide = wide_df.copy()
wide["easy_to_vhard_joint_drop"] = wide["very_hard_joint_correct"].fillna(0) - wide["easy_joint_correct"].fillna(0)
wide["easy_to_vhard_section_drop"] = wide["very_hard_section_correct"].fillna(0) - wide["easy_section_correct"].fillna(0)
wide["easy_to_vhard_epu_drop"] = wide["very_hard_epu_correct"].fillna(0) - wide["easy_epu_correct"].fillna(0)
most_harmed = wide.sort_values(["easy_to_vhard_joint_drop", "easy_to_vhard_section_drop", "easy_to_vhard_epu_drop"], ascending=[True, True, True]).head(100)
most_harmed_path = OUT_DIR / "q4_most_harmed_rows_upgraded.csv"
most_harmed.to_csv(most_harmed_path, index=False)
print(f"Saved hardest harmed rows -> {most_harmed_path}")

# ---------------------------
# MANIFEST
# ---------------------------
manifest = pd.DataFrame({
    "file": [
        prepared_path.name,
        wide_path.name,
        long_path.name,
        summary_overall_path.name,
        retrieval_table_path.name,
        pairwise_path.name,
        "q4_summary_by_gold_epu_upgraded.csv",
        "q4_summary_by_disagreement_upgraded.csv",
        "q4_summary_by_truncation_upgraded.csv",
        most_harmed_path.name,
        "run_status_q4_upgraded.csv",
    ]
})
manifest_path = OUT_DIR / "q4_manifest_upgraded.csv"
manifest.to_csv(manifest_path, index=False)
print(f"Saved manifest -> {manifest_path}")

print("\nDone. Files written to:")
for p in sorted(OUT_DIR.glob("*.csv")):
    print(" -", p)
